# Lab 4: Bandit challenge

You write one agent, which plays three bandit problems (the tracks) without knowing in
advance which one it faces. The agent follows the interface of Lab 3: `act(features)`
returns the arm to play, given the array of the feature vectors of the arms, and
`update(x, arm, reward)` receives the result. All rewards are Bernoulli.

The score of a track is the mean final pseudo-regret of your agent divided by the one of
UCB1, which ignores the features, on the same instances, in percent: UCB1 scores $100$, and
the lower the better. The score of the challenge is the mean of the three track scores.

In [ ]:
import numpy as np

from seqlearn.challenge import HORIZON, TRACKS, evaluate, load_agent, make_env

## 1. The tracks

Each track is a distribution over instances: `make_env(track, rng)` draws one. The horizon
is $T = 5000$ on every track.

- `slots`: $K = 20$ unrelated arms with small means, like click rates. The feature vector
  of arm $a$ is $e_a$: this is a $K$-armed bandit written as a linear bandit (Tutorial 3,
  exercise 3, question 1).
- `catalog`: $K = 300$ items described by $d = 8$ features, the same at every round, as
  the catalog of Lab 3.
- `users`: $K = 6$ items. At each round a new user arrives, with a profile over $4$ moods,
  and the feature vectors of the items depend on this profile ($d = 24$). The best item
  changes from one round to the next.

In [ ]:
for track in TRACKS:
    env = make_env(track, 0)
    features = env.features()
    means = features @ env.theta
    print(f"{track:8s} features {features.shape}, mean rewards: best {means.max():.3f}, "
          f"median {np.median(means):.3f}")

env = make_env("users", 0)
for t in range(3):
    print(f"users, round {t + 1}: best item {np.argmax(env.means())}, means {env.means().round(2)}")
    env.pull(0)

## 2. The reference

An agent is a file that defines a function `make_agent(n_arms, dim, horizon, rng)`. The
cell below writes the file of the reference, UCB1 on unrelated arms, with the tools of
`seqlearn`. Then `load_agent` reads the file again and `evaluate` plays $10$ runs on each
track. By construction, the reference scores $100$ on each track.

In [ ]:
%%writefile agent_ucb1.py
from seqlearn.agents import UCBAgent
from seqlearn.linear import ContextFree


def make_agent(n_arms, dim, horizon, rng):
    return ContextFree(UCBAgent(n_arms, 2.0, rng))

In [ ]:
evaluate(load_agent("agent_ucb1.py"));

## 3. Your agent

Rename the file in the first line of the cell below, and in the call to `load_agent` of
the next cell, to `agent_` followed by your names, for instance `agent_dupont_martin.py`,
and write your agent. The cell must contain
everything the agent needs: the imports, the classes (you may copy those of Lab 3) and
the function `make_agent`.

A few leads, from the simplest to the most ambitious:

- the agents of Lab 3 play the three tracks without any change;
- the parameters $\beta$ and $v$ were tuned on the catalog of Lab 3, not on these tracks;
- on `slots`, the means are small: the uniform prior of Thompson sampling, or the bonus of
  UCB1, which is designed for rewards in $[0, 1]$, may be far from the right scale;
- `make_agent` receives `n_arms` and `dim`, and the feature vectors tell the agent which
  structure it faces.

In [ ]:
%%writefile agent_yourname.py
import numpy as np

from seqlearn.linear import LinearAgent


class MyAgent(LinearAgent):
    def __init__(self, dim, rng=None):
        super().__init__(dim, rng)

    def reset(self):
        super().reset()
        ...  # TODO

    def act(self, features):
        ...  # TODO

    def update(self, x, arm, reward):
        super().update(x, arm, reward)
        ...  # TODO


def make_agent(n_arms, dim, horizon, rng):
    return MyAgent(dim, rng)

In [ ]:
evaluate(load_agent("agent_yourname.py"));

## 4. Rules

Your agent, in its file `agent_....py`, must follow these rules:

1. the file defines `make_agent(n_arms, dim, horizon, rng)` and only imports `numpy` and
   `seqlearn`;
2. all the randomness of the agent comes from `rng`;
3. `evaluate` with its default parameters runs in less than $3$ minutes on your laptop;
4. the agent only learns from `features`, `x`, `arm` and `reward`: it has no access to
   the environment.